# Lab 6: Actor-Critic on LunarLanderContinuous-v2

## Design Constraints

- A trained Agent must be saveable and restorable from disk (weights + config) so visualizations can be reproduced in the notebook without retraining.
- All hyperparameter choices (learning rates, architecture, activation functions, initial log-std, entropy regularization) must be documented in the notebook.

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Project Structure
# DONE: Establish /src, /test, /data, /img directory layout

! tree ..

..
├── data
│   ├── smoke_test_20260421_094513.csv
│   ├── test_run_20260421_093214.csv
│   ├── test_run_20260421_093215.csv
│   ├── test_run_20260421_093216.csv
│   ├── test_run_20260421_093249.csv
│   ├── test_run_20260421_093250.csv
│   ├── test_run_20260421_093251.csv
│   ├── test_run_20260421_093648.csv
│   ├── test_run_20260421_093649.csv
│   ├── test_run_20260421_094029.csv
│   ├── test_run_20260421_094058.csv
│   ├── test_run_20260421_094059.csv
│   ├── test_run_20260421_094100.csv
│   ├── test_run_20260421_094119.csv
│   ├── test_run_20260421_094139.csv
│   ├── test_run_20260421_094140.csv
│   ├── test_run_20260421_094141.csv
│   ├── test_run_20260421_094359.csv
│   ├── test_run_20260421_094400.csv
│   └── test_run_20260421_094401.csv
├── img
├── ipynb
│   ├── development.ipynb
│   └── report.ipynb
├── README.md
├── src
│   ├── __init__.py
│   ├── __pycache__
│   │   ├── __init__.cpython-311.pyc
│   │   ├── agent.cpython-311.pyc
│   │   └── trainer.cpython-311.pyc
│   ├── agen

In [3]:
# DONE: Verify PyTorch and Gymnasium are available; instantiate LunarLanderContinuous-v2
#       and inspect observation space, action space shape, and action bounds
import torch
import gymnasium as gym

print(f"PyTorch  : {torch.__version__}")
print(f"Gymnasium: {gym.__version__}")

env = gym.make("LunarLanderContinuous-v3")
obs_space = env.observation_space
act_space = env.action_space

print(f"\nObservation space : {obs_space}")
print(f"Action space      : {act_space}")
print(f"Action shape      : {act_space.shape}")
print(f"Action low        : {act_space.low}")
print(f"Action high       : {act_space.high}")
env.close()

PyTorch  : 2.10.0
Gymnasium: 1.2.3

Observation space : Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Action space      : Box(-1.0, 1.0, (2,), float32)
Action shape      : (2,)
Action low        : [-1. -1.]
Action high       : [1. 1.]


### Actor Hyperparameters

| Parameter | Value | Notes |
|---|---|---|
| `obs_dim` | 8 | LunarLanderContinuous-v3 observation space |
| `act_dim` | 2 | Main engine throttle, side engine throttle |
| `hidden_sizes` | `(64, 64)` | Two hidden layers; starting point for empirical tuning |
| `activation` | `ReLU` | Fast training; Tanh vs ReLU is a candidate experiment |
| `state_dependent_std` | `False` | Free-parameter mode; single global log-std shared across all states |
| Initial `log_std` | `0.0` | `exp(0) = 1.0`; broad initial exploration |
| `log_std` clamp | `[-20, 2]` | Prevents numerical instability from extreme variance estimates |

In [ ]:
# Actor
# DONE: Actor produces a Gaussian policy (mean and log-std) over continuous actions
# DONE: Tests for Actor

import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import gymnasium as gym
from torch.distributions import Normal
from src.agent import Actor

# --- environment bounds ---
env = gym.make("LunarLanderContinuous-v3")
obs_dim  = env.observation_space.shape[0]
act_dim  = env.action_space.shape[0]
act_low  = torch.tensor(env.action_space.low)
act_high = torch.tensor(env.action_space.high)
env.close()

torch.manual_seed(42)
actor_head = Actor(obs_dim, act_dim, state_dependent_std=True)
actor_free = Actor(obs_dim, act_dim, state_dependent_std=False)

obs1 = torch.zeros(obs_dim)
obs2 = torch.ones(obs_dim)

# --- forward pass ---
mu, log_std = actor_head(obs1)
print("=== Actor (state-dependent std) ===")
print(f"mu shape     : {mu.shape}   values: {mu.detach().numpy().round(3)}")
print(f"log_std shape: {log_std.shape}   values: {log_std.detach().numpy().round(3)}")

# --- sample, clip, log_prob, entropy ---
dist   = Normal(mu, log_std.exp())
action = dist.sample()
action_clipped = torch.clamp(action, act_low, act_high)
log_prob = dist.log_prob(action).sum(dim=-1)
entropy  = dist.entropy().sum(dim=-1)

print(f"\nRaw sample   : {action.detach().numpy().round(3)}")
print(f"Clipped      : {action_clipped.detach().numpy().round(3)}  (bounds [{act_low.numpy()}, {act_high.numpy()}])")
print(f"log_prob     : {log_prob.item():.4f}")
print(f"entropy      : {entropy.item():.4f}")

# --- mode contrast: does log_std vary with state? ---
print("\n=== Mode contrast ===")
_, log_std_head_obs1 = actor_head(obs1)
_, log_std_head_obs2 = actor_head(obs2)
_, log_std_free_obs1 = actor_free(obs1)
_, log_std_free_obs2 = actor_free(obs2)

print(f"Head  log_std(obs1): {log_std_head_obs1.detach().numpy().round(4)}")
print(f"Head  log_std(obs2): {log_std_head_obs2.detach().numpy().round(4)}  <- varies with state")
print(f"Free  log_std(obs1): {log_std_free_obs1.detach().numpy().round(4)}")
print(f"Free  log_std(obs2): {log_std_free_obs2.detach().numpy().round(4)}  <- same regardless of state")

### Critic Hyperparameters

| Parameter | Value | Notes |
|---|---|---|
| `obs_dim` | 8 | LunarLanderContinuous-v3 observation space |
| `hidden_sizes` | `(64, 64)` | Two hidden layers; starting point for empirical tuning |
| `activation` | `ReLU` | Matches Actor; Tanh vs ReLU is a candidate experiment |
| Output | scalar V(s) | Single value estimate per observation |

In [ ]:
# Critic
# DONE: Critic estimates state value V(s)
# DONE: Tests for Critic

import sys
sys.path.insert(0, '..')

import torch
from src.agent import Critic

obs_dim = 8

torch.manual_seed(42)
critic = Critic(obs_dim)

# --- single observation ---
obs = torch.zeros(obs_dim)
v = critic(obs)
print("=== Critic ===")
print(f"V(s) shape  : {v.shape}  (scalar)")
print(f"V(s) value  : {v.item():.4f}")
print(f"grad_fn     : {v.grad_fn}")

# --- batch of observations ---
batch_obs = torch.randn(4, obs_dim)
v_batch = critic(batch_obs)
print(f"\nBatch V(s) shape : {v_batch.shape}")
print(f"Batch V(s) values: {v_batch.detach().numpy().round(4)}")

# --- values vary across states ---
obs_a = torch.zeros(obs_dim)
obs_b = torch.ones(obs_dim)
print(f"\nV(zeros) : {critic(obs_a).item():.4f}")
print(f"V(ones)  : {critic(obs_b).item():.4f}  <- different states, different values")

### Agent Hyperparameters

| Parameter | Value | Notes |
|---|---|---|
| `actor_lr` | `3e-4` | Adam learning rate for Actor; standard starting point |
| `critic_lr` | `1e-3` | Adam learning rate for Critic; higher than actor is common |
| `gamma` | `0.99` | Discount factor; strong preference for future rewards |
| `act_low` / `act_high` | `[-1, 1]` | LunarLanderContinuous-v3 action bounds |

In [ ]:
# Agent
# DONE: Agent composes Actor and Critic; supports action selection and per-step TD updates
# DONE: Agent clips sampled actions to environment bounds before interacting with the environment
# DONE: Agent can be saved and restored from disk (weights + config) to avoid retraining
# DONE: Tests for Agent

import sys
sys.path.insert(0, '..')

import torch
import gymnasium as gym
from src.agent import Agent

env = gym.make("LunarLanderContinuous-v3")
obs_dim  = env.observation_space.shape[0]
act_dim  = env.action_space.shape[0]
act_low  = torch.tensor(env.action_space.low)
act_high = torch.tensor(env.action_space.high)
env.close()

torch.manual_seed(42)
agent = Agent(
    obs_dim=obs_dim, act_dim=act_dim,
    actor_lr=3e-4, critic_lr=1e-3, gamma=0.99,
    act_low=act_low, act_high=act_high,
)

obs      = torch.zeros(obs_dim)
next_obs = torch.randn(obs_dim)

# --- select action ---
action, log_prob, value, entropy = agent.select_action(obs)
print("=== Agent.select_action ===")
print(f"action   : {action.detach().numpy().round(4)}  (clipped to bounds)")
print(f"log_prob : {log_prob.item():.4f}  (summed over action dims)")
print(f"value    : {value.item():.4f}  V(s)")
print(f"entropy  : {entropy.item():.4f}  policy entropy")

# --- one-step update ---
_, _, next_value, _ = agent.select_action(next_obs)
losses = agent.update(log_prob, value, next_value, reward=1.0, done=False)
print("\n=== Agent.update (one step) ===")
print(f"actor_loss  : {losses['actor_loss']:.4f}")
print(f"critic_loss : {losses['critic_loss']:.4f}")

# --- save and restore ---
import tempfile, os
with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "agent.pt")
    agent.save(path)
    from src.agent import Agent as AgentCls
    restored = AgentCls.load(path)
    torch.manual_seed(0)
    a1, _, _, _ = agent.select_action(obs)
    torch.manual_seed(0)
    a2, _, _, _ = restored.select_action(obs)
    print("\n=== Save / Restore ===")
    print(f"original action  : {a1.detach().numpy().round(4)}")
    print(f"restored action  : {a2.detach().numpy().round(4)}")
    print(f"weights match    : {torch.allclose(a1, a2)}")

### Trainer Hyperparameters

| Parameter | Value | Notes |
|---|---|---|
| `label` | user-defined string | Namespaces the run; appears in the CSV filename and as a column |
| `seed` | `42` (default) | Seeds `torch` and `env.reset` for reproducibility; pass `None` to disable |
| `num_episodes` | user-defined | Number of training episodes per run |
| CSV output | `/data/<label>_<timestamp>.csv` | One row per step; all config values as columns for pandas comparison |
| Trajectory columns | `x`, `y`, `angle` | State of the lander at each step (obs indices 0, 1, 4) |

In [ ]:
# Trainer
# DONE: Trainer runs the Actor-Critic loop on LunarLanderContinuous-v3
# DONE: Trainer collects step-level metrics (TD error, entropy) and episode-level metrics (return, trajectory)
# DONE: Each run is namespaced by label + timestamp; all config values appear as columns in the CSV for pandas comparison
# DONE: Tests for Trainer

import sys
sys.path.insert(0, '..')

import torch
import gymnasium as gym
from src.agent import Agent
from src.trainer import Trainer

env = gym.make("LunarLanderContinuous-v3")
obs_dim  = env.observation_space.shape[0]
act_dim  = env.action_space.shape[0]
act_low  = torch.tensor(env.action_space.low)
act_high = torch.tensor(env.action_space.high)

agent = Agent(
    obs_dim=obs_dim, act_dim=act_dim,
    actor_lr=3e-4, critic_lr=1e-3, gamma=0.99,
    act_low=act_low, act_high=act_high,
)

trainer = Trainer(agent, env, label="smoke_test", seed=42)
df = trainer.train(num_episodes=3)
env.close()

print(f"Rows           : {len(df)}")
print(f"Columns        : {list(df.columns)}")
print(f"\nEpisode summary:")
print(df.groupby("episode")[["episode_return", "episode_length", "entropy"]].first().round(4))
print(f"\nSample step rows:")
print(df[["episode", "step", "reward", "td_error", "entropy", "x", "y", "angle"]].head(5).round(4))

In [12]:
# Visualizations (each exported as PNG to /img)
# TODO: Return per episode
# TODO: Policy entropy over training
# TODO: TD error magnitude over training
# TODO: Sample trajectories from the learned policy
# TODO: Actor mean action across the state space